# Micro-milling audio check (simple): STFT, Mel, PCEN, CWT + basic metrics

Segmented, minimal code. Plots use **magma** colormap (purple → orange → yellow).

In [ ]:
import re
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import scipy.signal as sig
import soundfile as sf

plt.rcParams["figure.dpi"] = 120

## 1) Files

Put the recordings here. Width should be in the filename like `3_991mm_...`.

In [ ]:
FILE_PATHS = [
    "13_01/4mm_mit_Luft_front_2.flac"
]

TARGET_SR = 192000

In [ ]:
def parse_depth_mm(name: str) -> float:
    m = re.search(r"x\s*(\d+)\s*([,.])\s*(\d+)\s*mm", name, flags=re.IGNORECASE)
    if m:
        return float(f"{m.group(1)}.{m.group(3)}")

    m = re.search(r"(\d+)\s*([,.])\s*(\d+)\s*mm", name, flags=re.IGNORECASE)
    if m:
        return float(f"{m.group(1)}.{m.group(3)}")

    return np.nan


for f in FILE_PATHS:
    depth = parse_depth_mm(Path(f).name)
    print(f, "-> depth_mm:", depth)

In [ ]:
def parse_width_mm(name: str) -> float:
    m = re.search(r"(\d+)mm", name)
    if not m:
        return float("nan")
    return float(f"{m.group(1)}")


for p in FILE_PATHS:
    print(p, "-> width_mm:", parse_width_mm(Path(p).name))

## 2) Load audio (mono) + pick a simple stable segment

Simple approach:
- skip first `SKIP_START_S` seconds (startup)
- skip last `SKIP_END_S` seconds (end transient)

In [ ]:
SKIP_START_S = 0
SKIP_END_S = 0
STFT_NFFT = 4096


def load_mono(
    path: str,
    target_sr: int | None = None,
    *,
    allow_resample: bool = True,
    normalize: bool = True,
) -> tuple[np.ndarray, int]:
    y, sr = sf.read(path, always_2d=False)

    # mono
    if y.ndim > 1:
        y = y.mean(axis=1)

    y = y.astype(np.float32)

    # resample if requested
    if target_sr is not None and sr != target_sr:
        if not allow_resample:
            raise ValueError(f"{path}: expected {target_sr} Hz, got {sr} Hz")

        g = np.gcd(sr, target_sr)
        up = target_sr // g
        down = sr // g
        y = sig.resample_poly(y, up, down).astype(np.float32)
        sr = target_sr

    # normalize peak (optional)
    if normalize:
        y = y / (np.max(np.abs(y)) + 1e-12)

    return y, sr


def stable_segment(y, sr, skip_start_s=SKIP_START_S, skip_end_s=SKIP_END_S):
    s = int(skip_start_s * sr)
    e = int(len(y) - skip_end_s * sr)
    e = max(e, s + 1)
    return y[s:e], s, e


def compute_stft_mag(y_seg: np.ndarray, sr: int) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Returns:
        S : magnitude STFT, shape (freq, time)
        f : frequency axis
        t : time axis
    """
    if len(y_seg) == 0:
        raise ValueError("Segment is empty, cannot compute STFT.")

    # use existing notebook constants if present, otherwise fall back safely
    win_len = int(globals().get("STFT_WIN", STFT_NFFT))
    hop_len = int(globals().get("STFT_HOP", max(1, win_len // 4)))

    nperseg = min(win_len, len(y_seg))
    hop_len = min(hop_len, nperseg)
    noverlap = max(0, nperseg - hop_len)

    f, t, Zxx = sig.stft(
        y_seg,
        fs=sr,
        window="hann",
        nperseg=nperseg,
        noverlap=noverlap,
        nfft=STFT_NFFT,
        boundary=None,
        padded=False,
    )

    S = np.abs(Zxx).astype(np.float32)
    return S, f.astype(np.float32), t.astype(np.float32)


records = []
for p in FILE_PATHS:
    y, sr = load_mono(p, TARGET_SR, normalize=False)
    seg, s, e = stable_segment(y, sr)

    stft_S, stft_f, stft_t = compute_stft_mag(seg, sr)

    records.append(
        {
            "path": p,
            "name": Path(p).name,
            # "width_mm": parse_width_mm(Path(p).name),
            "depth_mm": parse_depth_mm(Path(p).name),
            "sr": sr,
            "y": y,
            "seg": seg,
            "seg_range_s": (s / sr, e / sr),
            "stft_S": stft_S,
            "stft_f": stft_f,
            "stft_t": stft_t,
        }
    )

for r in records:
    print(
        r["name"],
        # "| width:", r["width_mm"],
        "| depth:",
        r["depth_mm"],
        "| stable segment (s):",
        r["seg_range_s"],
        "| seg_len_s:",
        len(r["seg"]) / r["sr"],
        "| stft shape:",
        r["stft_S"].shape,
    )

## 3) Waveform snippet + RMS

In [ ]:
def rms(x):
    return float(np.sqrt(np.mean(x * x) + 1e-12))


def dbfs(x):
    return 20 * np.log10(x + 1e-12)


summary = []
for r in records:
    r_rms = rms(r["seg"])
    r_peak = float(np.max(np.abs(r["seg"])))
    summary.append(
        {
            "name": r["name"],
            "depth_mm": r["depth_mm"],
            "rms": r_rms,
            "rms_dbfs": dbfs(r_rms),
            "peak": r_peak,
            "peak_dbfs": dbfs(r_peak),
            "crest_factor": r_peak / (r_rms + 1e-12),
        }
    )

summary = sorted(summary, key=lambda d: d["depth_mm"])
for s in summary:
    print(
        f"{s['depth_mm']:.1f} mm | RMS {s['rms_dbfs']:.1f} dBFS | Peak {s['peak_dbfs']:.1f} dBFS | Crest {s['crest_factor']:.2f} | {s['name']}"
    )

In [ ]:
import matplotlib.pyplot as plt
import numpy as np


def moving_rms(x, sr, win_s=0.02):
    n = max(1, int(win_s * sr))
    w = np.ones(n, dtype=np.float32) / n
    return np.sqrt(sig.convolve(x * x, w, mode="same") + 1e-12)


def percentile(x, p):
    return float(np.percentile(x, p))


def metrics_from_env(env):
    env_db = 20 * np.log10(env + 1e-12)
    return {
        "env_med": percentile(env_db, 50),
        "env_p90": percentile(env_db, 90),
        "env_p99": percentile(env_db, 99),
        "env_iqr": percentile(env_db, 75) - percentile(env_db, 25),
        "env_p99_minus_med": percentile(env_db, 99) - percentile(env_db, 50),
    }


for r in sorted(records, key=lambda rr: rr["depth_mm"]):
    env = moving_rms(r["seg"], r["sr"], win_s=0.02)
    m = metrics_from_env(env)
    print(
        r["depth_mm"],
        "mm | med:",
        round(m["env_med"], 2),
        "| p99-med:",
        round(m["env_p99_minus_med"], 2),
        "| IQR:",
        round(m["env_iqr"], 2),
    )


plt.figure(figsize=(12, 4))
for r in sorted(records, key=lambda rr: rr["depth_mm"]):
    env = moving_rms(r["seg"], r["sr"], win_s=0.02)  # 20 ms window
    tt = np.arange(len(env)) / r["sr"]
    plt.plot(tt, 20 * np.log10(env + 1e-12), label=f"{r['depth_mm']}mm")
plt.xlabel("Time [s]")
plt.ylabel("Envelope [dBFS]")
plt.title("Moving RMS envelope (stable segment)")
plt.legend()
plt.tight_layout()

In [ ]:
def plot_env_offset(records, win_s=0.02, offset_db=2.0):
    plt.figure(figsize=(12, 4))
    for i, r in enumerate(sorted(records, key=lambda rr: rr["depth_mm"])):
        env = moving_rms(r["seg"], r["sr"], win_s=win_s)
        tt = np.arange(len(env)) / r["sr"]
        env_db = 20 * np.log10(env + 1e-12) + i * offset_db
        plt.plot(tt, env_db, label=f"{r['depth_mm']}mm (+{i * offset_db:.1f} dB)", linewidth=1.1)

    plt.xlabel("Time [s]")
    plt.ylabel("Envelope [dBFS] + offset")
    plt.title("Moving RMS envelope (offset view)")
    plt.legend()
    plt.tight_layout()


plot_env_offset(records, win_s=0.01, offset_db=2)

In [ ]:
def plot_env_zoom(records, win_s=0.02, tmin=None, tmax=None, ymin=None, ymax=None):
    plt.figure(figsize=(12, 4))
    for r in sorted(records, key=lambda rr: rr["depth_mm"]):
        env = moving_rms(r["seg"], r["sr"], win_s=win_s)
        tt = np.arange(len(env)) / r["sr"]
        env_db = 20 * np.log10(env + 1e-12)

        m = np.ones_like(tt, dtype=bool)
        if tmin is not None:
            m &= tt >= tmin
        if tmax is not None:
            m &= tt <= tmax

        plt.plot(tt[m], env_db[m], label=f"{r['depth_mm']}mm", linewidth=1.2)

    if ymin is not None or ymax is not None:
        plt.ylim(ymin, ymax)
    if tmin is not None or tmax is not None:
        plt.xlim(tmin, tmax)

    plt.xlabel("Time [s]")
    plt.ylabel("Envelope [dBFS]")
    plt.title("Moving RMS envelope (zoom)")
    plt.legend()
    plt.tight_layout()

In [ ]:
plot_env_zoom(records, win_s=0.02, tmin=6.0, tmax=9.0, ymin=-33.5, ymax=-31.0)

plot_env_zoom(records, win_s=0.01, tmin=7.8, tmax=8.4, ymin=-33.5, ymax=-21.0)

In [ ]:
def band_rms(x, sr, f_lo=2000, f_hi=8000, order=4):
    sos = sig.butter(order, [f_lo, f_hi], btype="bandpass", fs=sr, output="sos")
    xb = sig.sosfilt(sos, x)
    return rms(xb)


for r in records:
    br = band_rms(r["seg"], r["sr"], 500, 2500)
    print(r["depth_mm"], "mm | band RMS dBFS:", 20 * np.log10(br + 1e-12))

In [ ]:
def rms(x):
    return float(np.sqrt(np.mean(x * x) + 1e-12))


def plot_wave_snip(y, sr, t0=1.0, dur=0.05, title=""):
    i0 = int(t0 * sr)
    i1 = int((t0 + dur) * sr)
    tt = np.arange(i0, i1) / sr
    plt.figure(figsize=(12, 3))
    plt.plot(tt, y[i0:i1], linewidth=0.8)
    plt.title(title)
    plt.xlabel("Time [s]")
    plt.ylabel("Amplitude")
    plt.tight_layout()


global_peak = max(np.max(np.abs(r["seg"])) for r in records)
ylim = (-1.05 * global_peak, 1.05 * global_peak)

for r in records:
    plot_wave_snip(r["y"], r["sr"], t0=1.0, dur=0.05, title=f"Wave snippet | {r['name']}")
    plt.ylim(*ylim)

In [ ]:
def bandpass(x, sr, f_lo=2000, f_hi=8000, order=4):
    sos = sig.butter(order, [f_lo, f_hi], btype="bandpass", fs=sr, output="sos")
    return sig.sosfilt(sos, x)


def plot_band_env(records, f_lo=2000, f_hi=8000, t0=1.33, dur=0.2, win_s=0.002):
    plt.figure(figsize=(12, 3))
    for r in sorted(records, key=lambda rr: rr["depth_mm"]):
        y, sr = r["y"], r["sr"]
        i0 = int(t0 * sr)
        i1 = int((t0 + dur) * sr)
        x = bandpass(y[i0:i1], sr, f_lo, f_hi)
        env = moving_rms(x, sr, win_s=win_s)
        env_db = 20 * np.log10(env + 1e-12)
        tt = np.arange(i0, i1) / sr
        plt.plot(tt, env_db, linewidth=1.2, label=f"{r['depth_mm']}mm")
    plt.xlabel("Time [s]")
    plt.ylabel("Band envelope [dBFS]")
    plt.title(f"Bandpass {f_lo}-{f_hi} Hz envelope (win={win_s * 1000:.1f} ms)")
    plt.legend()
    plt.tight_layout()


plot_band_env(records, f_lo=2500, f_hi=7500)

In [ ]:
def plot_amp_hist(records, t0=1.0, dur=0.5, bins=200):
    plt.figure(figsize=(12, 4))
    for r in sorted(records, key=lambda rr: rr["depth_mm"]):
        y, sr = r["y"], r["sr"]
        i0 = int(t0 * sr)
        i1 = int((t0 + dur) * sr)
        x = y[i0:i1]
        plt.hist(
            x, bins=bins, density=True, histtype="step", linewidth=1.5, label=f"{r['depth_mm']}mm"
        )
    plt.xlabel("Amplitude")
    plt.ylabel("Density")
    plt.title("Amplitude distribution (same time window)")
    plt.legend()
    plt.tight_layout()


plot_amp_hist(records, t0=1.33, dur=1.5)

## 4) STFT spectrogram (fixed scaling + magma)

Set `FMAX_PLOT=96_000` if full Nyquist at 192 kHz.

In [ ]:
STFT_NFFT = 8192
STFT_HOP = 2048

FMAX_PLOT = 20000
VMIN_DB, VMAX_DB = -120, -20


def relative_spec(S_db):
    # remove per-frequency baseline (median over time)
    return S_db - np.median(S_db, axis=1, keepdims=True)


def stft_mag_db(y, sr, nperseg=8192, hop=2048, nfft=None):
    noverlap = nperseg - hop
    f, t, Z = sig.stft(
        y,
        fs=sr,
        window="hann",
        nperseg=nperseg,
        noverlap=noverlap,
        nfft=nfft,  # <= can be > nperseg for zero-padding
        boundary=None,
        padded=False,
    )
    S = np.abs(Z) + 1e-12
    S_db = 20 * np.log10(S)
    return f, t, S_db, S


def plot_spec_db(f, t, S_db, title, fmax=FMAX_PLOT, vmin=VMIN_DB, vmax=VMAX_DB, cmap="magma"):
    fmax_eff = min(fmax, f[-1])
    m = f <= fmax_eff
    plt.figure(figsize=(12, 4))
    plt.pcolormesh(t, f[m], S_db[m, :], shading="auto", vmin=vmin, vmax=vmax, cmap=cmap)
    plt.ylim(0, fmax_eff)
    plt.xlabel("Time [s]")
    plt.ylabel("Freq [Hz]")
    plt.title(title)
    plt.colorbar(label="dB")
    plt.tight_layout()


for r in records:
    f, t, S_db, _ = stft_mag_db(r["seg"], r["sr"], nperseg=32768, hop=4096, nfft=32768)
    S_rel = relative_spec(S_db)
    plot_spec_db(
        f, t, S_rel, f"REL-STFT | {r['name']} | depth={r['depth_mm']}mm", vmin=-15, vmax=15
    )

## 5) Mel spectrogram + PCEN (magma)

Mel filterbank is implemented here (no extra libs).  
PCEN is a simple version suitable for comparing runs.

In [ ]:
N_MELS = 128
MEL_FMIN = 200
MEL_FMAX = 42400


def hz_to_mel(f):
    return 2595.0 * np.log10(1.0 + f / 700.0)


def mel_to_hz(m):
    return 700.0 * (10 ** (m / 2595.0) - 1.0)


def mel_filterbank(sr, n_fft, n_mels=128, fmin=0.0, fmax=None):
    if fmax is None:
        fmax = sr / 2
    f = np.linspace(0, sr / 2, n_fft // 2 + 1)
    mmin, mmax = hz_to_mel(fmin), hz_to_mel(fmax)
    m_pts = np.linspace(mmin, mmax, n_mels + 2)
    f_pts = mel_to_hz(m_pts)
    bins = np.floor((n_fft + 1) * f_pts / sr).astype(int)

    fb = np.zeros((n_mels, len(f)), dtype=np.float32)
    for i in range(1, n_mels + 1):
        left, center, right = bins[i - 1], bins[i], bins[i + 1]
        if center <= left:
            center = left + 1
        if right <= center:
            right = center + 1
        left = max(left, 0)
        right = min(right, len(f) - 1)
        for k in range(left, center):
            fb[i - 1, k] = (k - left) / (center - left + 1e-12)
        for k in range(center, right):
            fb[i - 1, k] = (right - k) / (right - center + 1e-12)

    fb = fb / (fb.sum(axis=1, keepdims=True) + 1e-12)
    return fb


def power_to_db(P):
    return 10 * np.log10(P + 1e-12)


def pcen(M, sr, hop, gain=0.98, bias=2.0, power=0.5, eps=1e-6, time_constant=0.06):
    s = 1.0 - np.exp(-(hop / sr) / max(time_constant, 1e-6))
    E = np.zeros_like(M, dtype=np.float32)
    E[:, 0] = M[:, 0]
    for ti in range(1, M.shape[1]):
        E[:, ti] = (1.0 - s) * E[:, ti - 1] + s * M[:, ti]
    out = (M / (eps + E) ** gain + bias) ** power - bias**power
    return out


for r in records:
    S = r["stft_S"]
    P = (S**2).astype(np.float32)  # (freq, time)

    # Derive n_fft from actual stored STFT shape (robust vs Cell 19 redefining STFT_NFFT)
    n_fft_used = (S.shape[0] - 1) * 2
    fb = mel_filterbank(
        r["sr"], n_fft_used, n_mels=N_MELS, fmin=MEL_FMIN, fmax=min(MEL_FMAX, r["sr"] / 2)
    )
    melP = fb @ P  # shapes guaranteed to match
    mel_db = power_to_db(melP)

    # Infer actual hop from stored time axis
    hop_used = int((r["stft_t"][1] - r["stft_t"][0]) * r["sr"]) if len(r["stft_t"]) > 1 else 1024
    mel_pcen = pcen(melP, r["sr"], hop_used)
    mel_pcen_db = power_to_db(np.maximum(mel_pcen, 0.0))

    plt.figure(figsize=(12, 4))
    plt.pcolormesh(
        r["stft_t"], np.arange(N_MELS), mel_db, shading="auto", cmap="magma", vmin=-120, vmax=-20
    )
    plt.title(f"Mel spectrogram | {r['name']}")
    plt.xlabel("Time [s]")
    plt.ylabel("Mel bin")
    plt.colorbar(label="dB")
    plt.tight_layout()

    plt.figure(figsize=(12, 4))
    plt.pcolormesh(
        r["stft_t"], np.arange(N_MELS), mel_pcen_db, shading="auto", cmap="magma", vmin=-60, vmax=10
    )
    plt.title(f"PCEN (mel) | {r['name']}")
    plt.xlabel("Time [s]")
    plt.ylabel("Mel bin")
    plt.colorbar(label="dB (PCEN)")
    plt.tight_layout()

In [ ]:
# ============================================================================
# 5.x) Combined stacked plot: STFT + Mel + PCEN
# Set the title template yourself.
# Available placeholders: {name}, {depth_mm}
# ============================================================================

COMBINED_STACKED_TITLE = "Spektrogrammen | 4mm Nut | Bearbeitung mit Kühlung"
COMBINED_FMAX_HZ = FMAX_PLOT  # e.g. 20000


def plot_combined_spectrogram_stack(
    stft_t,
    stft_f,
    stft_rel_db,
    mel_db,
    mel_pcen_db,
    *,
    title,
    fmax_hz=20000,
    stft_vmin=-15,
    stft_vmax=15,
    mel_vmin=-120,
    mel_vmax=-20,
    pcen_vmin=-60,
    pcen_vmax=10,
    cmap="magma",
):
    fmax_eff = min(fmax_hz, stft_f[-1])
    m = stft_f <= fmax_eff

    fig, axes = plt.subplots(3, 1, figsize=(7, 6), sharex=True)

    # 1) STFT
    im0 = axes[0].pcolormesh(
        stft_t,
        stft_f[m],
        stft_rel_db[m, :],
        shading="auto",
        cmap=cmap,
        vmin=stft_vmin,
        vmax=stft_vmax,
    )
    axes[0].set_ylabel("Freq [Hz]")
    axes[0].set_ylim(0, fmax_eff)
    axes[0].set_title("STFT")
    fig.colorbar(im0, ax=axes[0], label="dB")

    # 2) Mel
    im1 = axes[1].pcolormesh(
        stft_t,
        np.arange(mel_db.shape[0]),
        mel_db,
        shading="auto",
        cmap=cmap,
        vmin=mel_vmin,
        vmax=mel_vmax,
    )
    axes[1].set_ylabel("Mel bin")
    axes[1].set_title("Mel Spektrogramm")
    fig.colorbar(im1, ax=axes[1], label="dB")

    # 3) PCEN
    im2 = axes[2].pcolormesh(
        stft_t,
        np.arange(mel_pcen_db.shape[0]),
        mel_pcen_db,
        shading="auto",
        cmap=cmap,
        vmin=pcen_vmin,
        vmax=pcen_vmax,
    )
    axes[2].set_xlabel("Zeit [s]")
    axes[2].set_ylabel("Mel bin")
    axes[2].set_title("PCEN (mel)")
    fig.colorbar(im2, ax=axes[2], label="dB (PCEN)")

    fig.suptitle(title, fontsize=14, x=0.5, ha="center")
    fig.tight_layout(rect=[0, 0, 1, 0.97])


for r in records:
    # STFT relative dB from the stored STFT magnitude
    stft_db = 20 * np.log10(r["stft_S"] + 1e-12)
    stft_rel_db = relative_spec(stft_db)

    S = r["stft_S"]
    P = (S**2).astype(np.float32)

    n_fft_used = (S.shape[0] - 1) * 2
    fb = mel_filterbank(
        r["sr"],
        n_fft_used,
        n_mels=N_MELS,
        fmin=MEL_FMIN,
        fmax=min(MEL_FMAX, r["sr"] / 2),
    )

    melP = fb @ P
    mel_db = power_to_db(melP)

    hop_used = int((r["stft_t"][1] - r["stft_t"][0]) * r["sr"]) if len(r["stft_t"]) > 1 else 1024
    mel_pcen = pcen(melP, r["sr"], hop_used)
    mel_pcen_db = power_to_db(np.maximum(mel_pcen, 0.0))

    this_title = COMBINED_STACKED_TITLE.format(
        name=r["name"],
        depth_mm=r["depth_mm"],
    )

    plot_combined_spectrogram_stack(
        r["stft_t"],
        r["stft_f"],
        stft_rel_db,
        mel_db,
        mel_pcen_db,
        title=this_title,
        fmax_hz=COMBINED_FMAX_HZ,
    )

## 6) Continuous Wavelet Transform (CWT) - Morlet Wavelet

The CWT provides a time-frequency representation with better resolution control than STFT.  
Using the Morlet wavelet (complex exponential modulated by a Gaussian), we can see how 
different frequency components evolve over time at different milling depths.

**Key Parameters to Adjust:**
- `w` (omega0): Morlet wavelet width parameter - controls time vs frequency resolution
  - Higher `w` (e.g., 12-20): Better frequency resolution, worse time resolution
  - Lower `w` (e.g., 4-8): Better time resolution, worse frequency resolution
  - Default is typically 5-6
- `n_scales`: Number of frequency scales to compute
  - More scales = smoother frequency axis but slower computation
  - Typical range: 100-500
- `f_min`, `f_max`: Frequency range to analyze
  - Adjust based on signal of interest
- `segment_duration`: Length of audio segment to analyze (in seconds)
  - Shorter = faster computation, longer = more context

In [ ]:
# ============================================================================
# CWT PARAMETERS
# ============================================================================

# Morlet wavelet width parameter
# Higher w = better frequency resolution, worse time resolution
# Lower w = better time resolution, worse frequency resolution
CWT_W = 10.0

# Frequency range to analyze
CWT_F_MIN = 100  # Minimum frequency (Hz) - adjust to focus on the band of interest
CWT_F_MAX = 40000  # Maximum frequency (Hz) - can go up to Nyquist (96 kHz)

# Number of frequency scales (more = smoother frequency axis)
N_SCALES_PER_CHUNK = 15
CWT_N_SCALES = 200  # Try 100-500 depending on desired resolution vs computation time

# Time segment to analyze (set to None to use full segment)
CWT_T_START = 0  # Start time in seconds (from beginning of stable segment)
CWT_DURATION = 2  # Duration in seconds (None = use all available)

# Visualization parameters
CWT_VMIN_DB = -60  # Minimum dB for colormap (adjust to see more/less detail)
CWT_VMAX_DB = -20  # Maximum dB for colormap

print("CWT Configuration:")
print(f"  Morlet width (w): {CWT_W}")
print(f"  Frequency range: {CWT_F_MIN}-{CWT_F_MAX} Hz")
print(f"  Number of scales: {CWT_N_SCALES}")
print(f"  Time segment: {CWT_T_START}s + {CWT_DURATION}s")
print(f"  dB range: [{CWT_VMIN_DB}, {CWT_VMAX_DB}]")

In [ ]:
def compute_cwt_morlet(y, sr, w=6.0, f_min=500, f_max=40000, n_scales=250, scales_per_chunk=25):
    """
    Compute Continuous Wavelet Transform using Morlet wavelet.

    Memory-efficient implementation that processes signal in chunks.
    The Morlet wavelet is a complex exponential (carrier wave) modulated by
    a Gaussian window. It provides good joint time-frequency localization.

    Parameters:
    -----------
    y : array
        Input signal
    sr : int
        Sample rate
    w : float
        Morlet wavelet width parameter (omega0)
        - Controls the number of oscillations in the wavelet
        - Higher values give better frequency resolution
        - Typical range: 4-20
    f_min, f_max : float
        Frequency range to analyze
    n_scales : int
        Number of scales (frequencies) to compute
    chunk_size : int
        Number of samples to process at once (for memory efficiency)
        Smaller values use less memory but may be slightly slower
        Default 50000 works well for most cases

    Returns:
    --------
    cwt : 2D array
        CWT coefficients (complex values)
    freqs : 1D array
        Frequencies corresponding to each scale
    t : 1D array
        Time axis
    """

    # Generate frequency array (log-spaced for better coverage)
    freqs = np.logspace(np.log10(f_min), np.log10(f_max), n_scales)

    # Time axis
    N = len(y)
    t = np.arange(N) / sr

    # Process in chunks to save memory
    # We'll compute a few scales at a time instead of all at once
    n_chunks = int(np.ceil(n_scales / scales_per_chunk))

    # Initialize output array (we'll fill it chunk by chunk)
    cwt_matrix = np.zeros((n_scales, N), dtype=complex)

    print(f"  Computing CWT in {n_chunks} frequency chunks...", end="", flush=True)

    for chunk_idx in range(n_chunks):
        # Determine which scales to process in this chunk
        scale_start = chunk_idx * scales_per_chunk
        scale_end = min((chunk_idx + 1) * scales_per_chunk, n_scales)
        chunk_freqs = freqs[scale_start:scale_end]

        # Process these scales
        for local_i, freq in enumerate(chunk_freqs):
            global_i = scale_start + local_i

            # Create Morlet wavelet at this frequency
            n_cycles = w
            wavelet_duration = n_cycles / freq
            wavelet_samples = int(wavelet_duration * sr)
            if wavelet_samples % 2 == 0:
                wavelet_samples += 1
            wavelet_samples = min(wavelet_samples, N)

            # Create time vector for wavelet (centered at 0)
            half_width = wavelet_samples // 2
            t_wavelet = np.arange(-half_width, half_width + 1) / sr

            # Standard deviation of Gaussian envelope
            sigma = n_cycles / (2 * np.pi * freq)

            # Create complex Morlet wavelet
            carrier = np.exp(2j * np.pi * freq * t_wavelet)
            envelope = np.exp(-(t_wavelet**2) / (2 * sigma**2))
            wavelet = carrier * envelope

            # Normalize wavelet energy
            wavelet = wavelet / np.sqrt(np.sum(np.abs(wavelet) ** 2))

            # Convolve signal with wavelet
            cwt_matrix[global_i, :] = np.convolve(y, np.conj(wavelet), mode="same")

        # Progress indicator
        if (chunk_idx + 1) % max(1, n_chunks // 4) == 0:
            print(f" {int(100 * (chunk_idx + 1) / n_chunks)}%", end="", flush=True)

    print(" Done!")
    return cwt_matrix, freqs, t


def cwt_to_db(cwt_matrix, ref_max=False):
    """
    Convert complex CWT coefficients to dB magnitude.

    Parameters:
    -----------
    cwt_matrix : 2D complex array
        CWT coefficients
    ref_max : bool
        If True, reference to maximum value (relative dB)
        If False, use absolute magnitude

    Returns:
    --------
    cwt_db : 2D array
        Magnitude in dB
    """
    # Take absolute value (magnitude) of complex coefficients
    magnitude = np.abs(cwt_matrix)

    # Avoid log(0) by adding small epsilon
    epsilon = 1e-12
    magnitude = magnitude + epsilon

    # Convert to dB
    cwt_db = 20 * np.log10(magnitude)

    # Optionally reference to maximum
    if ref_max:
        cwt_db = cwt_db - np.max(cwt_db)

    return cwt_db


def plot_cwt(freqs, t, cwt_db, title, vmin=-60, vmax=0, cmap="magma"):
    """
    Plot CWT spectrogram with proper frequency axis scaling.
    """
    plt.figure(figsize=(12, 4))

    # Use pcolormesh for plotting (shading='auto' for better rendering)
    plt.pcolormesh(t, freqs, cwt_db, shading="auto", vmin=vmin, vmax=vmax, cmap=cmap)

    # Use log scale for frequency axis (since we used logspace)
    plt.yscale("log")

    # Format frequency axis nicely
    plt.ylabel("Frequency [Hz]")
    plt.xlabel("Time [s]")
    plt.title(title)

    # Add colorbar
    cbar = plt.colorbar(label="Magnitude [dB]")

    plt.tight_layout()
    plt.show()


print("CWT functions defined (memory-efficient chunked implementation).")
print("This version processes frequency scales in small batches to avoid memory issues.")

In [ ]:
# ============================================================================
# COMPUTE AND PLOT CWT FOR ALL DEPTHS
# ============================================================================

print("Computing CWT for all depth levels...\n")

cwt_results = []  # Store results for later comparison

for r in sorted(records, key=lambda x: x["depth_mm"]):
    depth = r["depth_mm"]
    seg = r["seg"]
    sr = r["sr"]
    name = r["name"]

    # Extract time segment if specified
    if CWT_T_START is not None or CWT_DURATION is not None:
        i_start = int(CWT_T_START * sr) if CWT_T_START is not None else 0

        if CWT_DURATION is not None:
            i_end = i_start + int(CWT_DURATION * sr)
        else:
            i_end = len(seg)

        # Make sure we don't exceed array bounds
        i_start = max(0, min(i_start, len(seg)))
        i_end = max(i_start, min(i_end, len(seg)))

        y_segment = seg[i_start:i_end]
        time_offset = CWT_T_START if CWT_T_START is not None else 0
    else:
        y_segment = seg
        time_offset = 0

    print(f"Processing depth {depth} mm... ({len(y_segment) / sr:.2f} seconds)")

    # Compute CWT
    cwt_matrix, freqs, t = compute_cwt_morlet(
        y_segment,
        sr,
        w=CWT_W,
        f_min=CWT_F_MIN,
        f_max=CWT_F_MAX,
        n_scales=CWT_N_SCALES,
        scales_per_chunk=N_SCALES_PER_CHUNK,
    )

    # Convert to dB (referenced to maximum for relative comparison)
    cwt_db = cwt_to_db(cwt_matrix, ref_max=True)

    # Adjust time axis for offset
    t_plot = t + time_offset

    # Store results
    cwt_results.append(
        {"depth_mm": depth, "name": name, "cwt_db": cwt_db, "freqs": freqs, "t": t_plot, "sr": sr}
    )

    # Plot
    title = f"CWT (Morlet, w={CWT_W}) | {name} | depth={depth} mm"
    plot_cwt(freqs, t_plot, cwt_db, title, vmin=CWT_VMIN_DB, vmax=CWT_VMAX_DB, cmap="magma")

print(f"\nCWT computation complete for {len(cwt_results)} depths.")

### 6.1) CWT Comparison: Average Frequency Content vs Depth

Here we compute the time-averaged CWT magnitude for each depth to see
which frequencies are most prominent at different cutting depths.

In [ ]:
# ============================================================================
# COMPARE AVERAGE FREQUENCY CONTENT ACROSS DEPTHS
# ============================================================================

plt.figure(figsize=(12, 6))

for res in cwt_results:
    # Compute time-averaged magnitude (mean along time axis)
    avg_magnitude = np.mean(res["cwt_db"], axis=1)

    # Plot on log frequency scale
    plt.plot(res["freqs"], avg_magnitude, label=f"{res['depth_mm']} mm", linewidth=2, alpha=0.8)

plt.xscale("log")
plt.xlabel("Frequency [Hz]")
plt.ylabel("Average CWT Magnitude [dB]")
plt.title("Time-Averaged CWT Magnitude vs Frequency (all depths)")
plt.grid(True, which="both", alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

print("Comparison plot complete.")
print("\nObservations to look for:")
print("- Do deeper cuts show more energy at specific frequencies?")
print("- Are there resonant frequencies that change with depth?")
print("- How does the high-frequency content (>10kHz) vary with depth?")

### 6.2) Temporal Energy Evolution in Specific Frequency Bands

Extract how energy evolves over time in specific frequency bands of interest.

In [ ]:
# ============================================================================
# FREQUENCY BANDS TO ANALYZE (adjust these to the bands of interest)
# ============================================================================

freq_bands = [
    ("Low", 500, 2000),  # Low frequency band
    ("Mid", 2000, 8000),  # Mid frequency band
    ("High", 8000, 20000),  # High frequency band
    ("Ultra", 20000, 40000),  # Ultra-high frequency band
]

# Create subplots for each band
fig, axes = plt.subplots(len(freq_bands), 1, figsize=(12, 3 * len(freq_bands)), sharex=True)

for ax_idx, (band_name, f_low, f_high) in enumerate(freq_bands):
    ax = axes[ax_idx] if len(freq_bands) > 1 else axes

    for res in cwt_results:
        freqs = res["freqs"]
        cwt_db = res["cwt_db"]
        t = res["t"]

        # Find frequency indices in this band
        band_mask = (freqs >= f_low) & (freqs <= f_high)

        if np.any(band_mask):
            # Average over frequency dimension within this band
            band_energy = np.mean(cwt_db[band_mask, :], axis=0)

            # Plot temporal evolution
            ax.plot(t, band_energy, label=f"{res['depth_mm']} mm", linewidth=1.5, alpha=0.8)

    ax.set_ylabel(f"{band_name}\n{f_low}-{f_high} Hz\n[dB]")
    ax.set_title(f"Temporal Energy Evolution: {band_name} Band ({f_low}-{f_high} Hz)")
    ax.legend(loc="upper right", ncol=2)
    ax.grid(True, alpha=0.3)

axes[-1].set_xlabel("Time [s]")
plt.tight_layout()
plt.show()

print("Temporal energy evolution plots complete.")
print("\nLook for:")
print("- Transient events (spikes) that correlate with depth")
print("- Sustained differences in energy levels between depths")
print("- Frequency bands where depth has the strongest effect")